# Demo 2: Watch the Interval Collapse (STUDENT)

**Module 01, Section 5. Follow along with the instructor.**

Eight minutes. You will build a bootstrap confidence interval function, run
it on the full Cordwell test set, then on a 200 review slice, then across
test sets of five different sizes, and plot how the interval width collapses
as the number of positive examples grows.

Cells marked **YOUR TURN** have a short piece of code for you to write. The
contract is in the comment. Everything else is pre written. Run the last
checks cell at any time; a fresh notebook passes 1 of 12, and that is
expected. The sweep cell takes about 14 seconds when it runs for real; the
rows print one at a time as they finish, so a quiet pause is normal.


## Setup

Imports and data load. Nothing to write here. Same test set as Demo 1:
2,000 synthetic Cordwell reviews, 67 genuine safety escalations, and the
model's score per review. Predictions are cut at the same 0.50 threshold as
this morning, which produced the F1 of 0.4670 that this demo is about to
put error bars on.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("cordwell_test_set.csv")
y_test = df["y_true"].to_numpy()
y_score = df["y_score"].to_numpy()
y_pred = (y_score >= 0.50).astype(int)

print(f"{len(df)} reviews, {int(y_test.sum())} positives")
print(f"F1 at threshold 0.50: {f1_score(y_test, y_pred):.4f}")


## Beat 1: The thirteen line bootstrap

The question on the slide: 0.4670 came from 67 positive examples. Collect a
different 67 from the same population and the number moves. How far?

The bootstrap answers it by simulation. We cannot re collect the test set,
but we can fake it: draw 2,000 rows **from our own test set, with
replacement**, so some rows appear twice and some not at all. Each such
resample is a plausible alternate test set. Score each one, do it 2,000
times, and the middle 95 percent of those scores is the confidence interval.

Two pre written details worth reading before you fill in the loop:

* The two `np.asarray` calls are not decoration. pandas Series index rows by
  label, numpy arrays by position, and the fancy indexing below needs
  positions. Without the coercion this function crashes on the output of
  `train_test_split`.
* A resample can, by bad luck, contain zero positives, and F1 is meaningless
  there. Those degenerate resamples are skipped.


In [ ]:
def bootstrap_ci(y_true, y_pred, metric, n_boot=2000, seed=0):
    # Coerce first: pandas indexes by LABEL, numpy by POSITION.
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rng = np.random.default_rng(seed)
    n, vals = len(y_true), []
    for _ in range(n_boot):
        # YOUR TURN (ST-1). Three lines:
        #   idx = a numpy array of n random row positions in [0, n),
        #         drawn WITH replacement: rng.integers(0, n, n)
        #   if the resampled truth y_true[idx] contains zero positives,
        #         continue (skip this resample)
        #   otherwise append metric(y_true[idx], y_pred[idx],
        #         zero_division=0) to vals
        ...
    return np.percentile(vals, [2.5, 97.5])

print("bootstrap_ci defined")


## Beat 2: The interval on the number we trusted

Run it on the full test set. Before executing: the slide asked you to guess
how far 0.4670 moves. Lock in a guess.


In [ ]:
# YOUR TURN (ST-2)
# Contract:
#   ci_full    = bootstrap_ci applied to y_test, y_pred, f1_score
#                (defaults are fine: 2000 resamples, seed 0)
#   width_full = upper end minus lower end of ci_full
ci_full = None
width_full = None

if ci_full is not None:
    print(f"F1 point estimate: {f1_score(y_test, y_pred):.4f}")
    print(f"95% interval:      [{ci_full[0]:.4f}, {ci_full[1]:.4f}]")
    print(f"Width:             {width_full:.4f}")


The number the room has been quoting as 0.4670 is, with 95 percent
confidence, somewhere between roughly 0.39 and 0.55. The first decimal is
solid. The second is already in doubt. The third and fourth were a costume.

That is with 67 positives, which sounded like plenty when the test set was
introduced as 2,000 rows this morning.


## Beat 3: The 200 row slice

A realistic proposal someone will make on your team: evaluate on a quick
200 review sample to save labeling budget. Take one (the sample seed is
pinned so the numbers match the slide) and put an interval on it.


In [ ]:
SLICE_SEED = 1066   # pinned: reproduces the slice on the slide exactly

slice_df = df.sample(n=200, random_state=SLICE_SEED)

# YOUR TURN (ST-3)
# Contract:
#   y_slice  = the slice's y_true column as a numpy array
#   p_slice  = predictions for the slice: its y_score column as a numpy
#              array, cut at 0.50, converted to int (same one liner as
#              Demo 1's ST-1)
#   ci_slice = bootstrap_ci on the slice
y_slice = None
p_slice = None
ci_slice = None

if ci_slice is not None:
    print(f"Rows: {len(y_slice)}   Positives: {int(y_slice.sum())}")
    print(f"F1 point estimate: {f1_score(y_slice, p_slice):.4f}")
    print(f"95% interval:      [{ci_slice[0]:.4f}, {ci_slice[1]:.4f}]")
    print(f"Width:             {ci_slice[1] - ci_slice[0]:.4f}")


Six positives. The interval runs from about 0.10 to 0.60, which spans
essentially the entire useful range for this task. An evaluation with that
interval cannot distinguish a good model from a bad one; it can only
distinguish a model from a coin. Nothing about the code was wrong. The
sample was simply too small, and only the interval says so out loud.

Note which number governed that: not the 200 rows, the 6 positives. With
rare positives, your effective test set size is the positive count.


## Beat 4: The collapse

Run the same measurement at five test set sizes: 100, 200, 500, 1,000,
2,000 rows. Each subset is sampled with stratification, the discipline from
this morning's split slide, so the 3.35 percent base rate survives at every
size and the positive count grows predictably: 3, 7, 17, 34, 67.

The `subset` helper below is pre written plumbing (it also handles the top
size, where the subset is just the whole set). Your loop does the science.
About 14 seconds of compute; the rows print as they land.


In [ ]:
def subset(frame, n_rows, seed=0):
    # Stratified subsample so the rare positive rate survives at every size.
    if n_rows >= len(frame):
        return frame
    sub, _ = train_test_split(
        frame,
        train_size=n_rows,
        stratify=frame["y_true"],
        random_state=seed,
    )
    return sub

print("subset helper defined")


In [ ]:
SWEEP_SIZES = [100, 200, 500, 1000, 2000]
sweep_rows = []

print(f"{'N':>5} {'pos':>4} {'F1':>6} {'lo':>7} {'hi':>7} {'width':>7}")
for N in SWEEP_SIZES:
    # YOUR TURN (ST-4). For each size N:
    #   sub    = subset(df, N)
    #   yt     = sub's y_true column as a numpy array
    #   yp     = sub's y_score column as a numpy array, cut at 0.50, as int
    #   lo, hi = bootstrap_ci(yt, yp, f1_score)   (it unpacks into two)
    #   append to sweep_rows a dict with keys:
    #       N, positives (int(yt.sum())), f1 (f1_score(yt, yp)),
    #       lo, hi, width (hi minus lo)
    ...
    if sweep_rows and sweep_rows[-1]["N"] == N:
        r = sweep_rows[-1]
        print(f"{r['N']:>5} {r['positives']:>4} {r['f1']:>6.3f} "
              f"{r['lo']:>7.3f} {r['hi']:>7.3f} {r['width']:>7.3f}")


The plot cell is pre written. Interval width on the y axis, positive count
on the x axis. The dashed line marks 67 positives: the room's own test set.


In [ ]:
if sweep_rows:
    xs = [r["positives"] for r in sweep_rows]
    ws = [r["width"] for r in sweep_rows]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(xs, ws, marker="o", linewidth=2)
    for r in sweep_rows:
        ax.annotate(f"N={r['N']}", (r["positives"], r["width"]),
                    textcoords="offset points", xytext=(8, 8))
    ax.axvline(67, linestyle="--", color="gray")
    ax.annotate("your test set\n(67 positives)", (67, ws[0] * 0.9),
                ha="right", xytext=(-8, 0), textcoords="offset points",
                color="gray")
    ax.set_xlabel("Positive examples in the test set")
    ax.set_ylabel("Width of the 95% interval on F1")
    ax.set_title("The interval collapses as positives accumulate")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("interval_collapse.png", dpi=150)
    plt.show()


Read the two ends of the curve to the room. At 3 positives the interval is
wider than half the metric's entire range: that evaluation is a shrug. At 67
positives the width is 0.16, which is enough to gate a release but nowhere
near enough to certify a fourth decimal place. And the curve is flattening:
the next big reduction in width costs a multiple of the labels already
collected, because width shrinks roughly with the square root of the
positive count. Labeling budget has diminishing returns, and this curve is
the planning tool for it.

The rule this buys, stated on the next slide: **no model comparison without
intervals.** Two F1 numbers 0.02 apart with 0.16 wide overlapping intervals
is not a measured difference; it is noise wearing a ranking.


## Checks

Run any time. Reads only what you have defined; never crashes on the rest.
The sweep checks re run nothing; they inspect `sweep_rows`.


In [ ]:
passed, total = 0, 0

def check(name, fn):
    global passed, total
    total += 1
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    mark = "PASS" if ok else "not yet"
    passed += ok
    print(f"[{mark:>7}] {name}")

def _fixture_ok():
    yt = np.array([1, 0, 1, 0, 1, 0, 0, 0])
    yp = np.array([1, 0, 0, 0, 1, 0, 1, 0])
    ci = bootstrap_ci(yt, yp, f1_score, n_boot=200, seed=7)
    return len(ci) == 2 and 0 <= ci[0] <= ci[1] <= 1

check("data loaded (2000 rows, 67 positives)",
      lambda: len(df) == 2000 and int(y_test.sum()) == 67)
check("ST-1 bootstrap_ci returns an ordered pair on a tiny fixture",
      _fixture_ok)
check("ST-2 full set lower bound is 0.3871",
      lambda: abs(ci_full[0] - 0.3871) < 5e-4)
check("ST-2 full set upper bound is 0.5487",
      lambda: abs(ci_full[1] - 0.5487) < 5e-4)
check("ST-2 full set width is 0.1616",
      lambda: abs(width_full - 0.1616) < 5e-4)
check("ST-3 slice has 200 rows and 6 positives",
      lambda: len(y_slice) == 200 and int(y_slice.sum()) == 6)
check("ST-3 slice F1 is 0.3636",
      lambda: abs(f1_score(y_slice, p_slice) - 4 / 11) < 5e-4)
check("ST-3 slice interval is [0.0952, 0.6000]",
      lambda: abs(ci_slice[0] - 0.0952) < 5e-4
              and abs(ci_slice[1] - 0.6000) < 5e-4)
check("ST-4 sweep has 5 rows with positives 3, 7, 17, 34, 67",
      lambda: [r["positives"] for r in sweep_rows] == [3, 7, 17, 34, 67])
check("ST-4 widths strictly decrease as positives grow",
      lambda: len(sweep_rows) == 5
              and all(sweep_rows[i]["width"] > sweep_rows[i + 1]["width"]
                  for i in range(len(sweep_rows) - 1)))
check("ST-4 the widest interval is at least 3x the narrowest",
      lambda: sweep_rows[0]["width"] > 3 * sweep_rows[-1]["width"])
check("ST-4 the N=2000 row agrees with the ST-2 interval",
      lambda: abs(sweep_rows[-1]["width"] - width_full) < 1e-9)

print(f"\n{passed} of {total} checks passing")
